In [1]:
#import libraries
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

I0000 00:00:1783573940.512335   75331 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1783573940.512866   75331 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1783573940.554078   75331 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1783573941.523267   75331 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [2]:
#load data
four_df = pd.read_csv('/home/yvonne_chook/github/207-Summer26-FinalProject-MLModel/Merged_EDA/combined_data/combined_data_4h.csv')

one_df = pd.read_csv('/home/yvonne_chook/github/207-Summer26-FinalProject-MLModel/Merged_EDA/combined_data/combined_data_1h.csv')

- No delay or early departure (0)
- Less than 1 hour delay (1)
- 1–2 hour delay (2)
- 2–3 hour delay (3)
- 3–4 hour delay (4)
- 4–5 hour delay (5)
- 5–6 hour delay (6) 
- More than 6 hour delay (7)
- Flight cancelled (8)

Note to team, there should be a total of 9 outcomes

In [3]:
#currently only 8 displayed
four_df["outcome"].unique()

array([0., 1., 2., 8., 3., 4., 6., 5., 7.])

In [4]:
four_df

,date,scheduled_dep_dt,actual_dep_dt,carrier_code,destination_airport,scheduled_elapsed_time_minutes,year,month,day_of_week,is_weekend,...,dew_point_temperature,relative_humidity,altimeter,aircraft_age,week_num,temperature_dewpoint_spread,airport_delay_average_1h,airport_delay_stddev_1h,airport_departures_observed_1h,departure_delay_minutes
0,2022-01-01,2022-01-01 05:25:00,2022-01-01 05:25:00,WN,DEN,145.0,2022.0,1.0,5.0,1,...,5.0,69.0,1012.9,16.0,1,5.6,134.666667,30.746273,3.0,0.0
1,2022-01-01,2022-01-01 05:55:00,2022-01-01 05:52:00,DL,SLC,118.0,2022.0,1.0,5.0,1,...,5.0,69.0,1012.9,3.0,1,5.6,136.250000,24.931573,4.0,-3.0
2,2022-01-01,2022-01-01 06:00:00,2022-01-01 06:14:00,DL,ATL,272.0,2022.0,1.0,5.0,1,...,4.4,68.0,1013.2,20.0,1,5.6,136.250000,24.931573,4.0,14.0
3,2022-01-01,2022-01-01 06:02:00,2022-01-01 05:56:00,UA,LAX,98.0,2022.0,1.0,5.0,1,...,4.4,68.0,1013.2,6.0,1,5.6,141.666667,27.501515,3.0,-6.0
4,2022-01-01,2022-01-01 06:10:00,2022-01-01 07:51:00,DL,MSP,218.0,2022.0,1.0,5.0,1,...,4.4,68.0,1013.2,24.0,1,5.6,149.000000,26.820390,4.0,101.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
339991,2026-04-30,2026-04-30 15:17:00,2026-04-30 15:17:00,AS,LAX,92.0,2026.0,4.0,3.0,0,...,9.4,83.0,1015.6,17.0,18,2.8,14.444444,31.939045,27.0,0.0
339992,2026-04-30,2026-04-30 15:17:00,2026-04-30 15:21:00,AA,JFK,342.0,2026.0,4.0,3.0,0,...,9.4,83.0,1015.6,12.0,18,2.8,14.444444,31.939045,27.0,4.0
339993,2026-04-30,2026-04-30 15:20:00,2026-04-30 15:13:00,UA,ORD,273.0,2026.0,4.0,3.0,0,...,9.4,83.0,1015.6,13.0,18,2.8,15.720000,32.858941,25.0,-7.0
339994,2026-04-30,2026-04-30 15:40:00,2026-04-30 17:15:00,WN,SAN,95.0,2026.0,4.0,3.0,0,...,9.4,83.0,1015.6,1.0,18,2.8,21.363636,36.845295,22.0,95.0


In [5]:
one_df

,date,scheduled_dep_dt,actual_dep_dt,carrier_code,destination_airport,scheduled_elapsed_time_minutes,year,month,day_of_week,is_weekend,...,dew_point_temperature,relative_humidity,altimeter,aircraft_age,week_num,temperature_dewpoint_spread,airport_delay_average_1h,airport_delay_stddev_1h,airport_departures_observed_1h,departure_delay_minutes
0,2022-01-01,2022-01-01 05:25:00,2022-01-01 05:25:00,WN,DEN,145.0,2022.0,1.0,5.0,1,...,5.0,74.0,1014.6,16.0,1,4.4,0.000000,0.000000,0.0,0.0
1,2022-01-01,2022-01-01 05:55:00,2022-01-01 05:52:00,DL,SLC,118.0,2022.0,1.0,5.0,1,...,5.0,74.0,1014.6,3.0,1,4.4,0.000000,0.000000,0.0,-3.0
2,2022-01-01,2022-01-01 06:00:00,2022-01-01 06:14:00,DL,ATL,272.0,2022.0,1.0,5.0,1,...,5.0,77.0,1015.6,20.0,1,3.9,0.000000,0.000000,0.0,14.0
3,2022-01-01,2022-01-01 06:02:00,2022-01-01 05:56:00,UA,LAX,98.0,2022.0,1.0,5.0,1,...,5.0,77.0,1015.6,6.0,1,3.9,0.000000,0.000000,0.0,-6.0
4,2022-01-01,2022-01-01 06:10:00,2022-01-01 07:51:00,DL,MSP,218.0,2022.0,1.0,5.0,1,...,5.0,77.0,1015.6,24.0,1,3.9,0.000000,0.000000,0.0,101.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
334081,2026-04-30,2026-04-30 12:37:00,2026-04-30 13:17:00,DL,MSP,223.0,2026.0,4.0,3.0,0,...,9.4,83.0,1015.6,9.0,18,2.8,18.904762,36.750381,21.0,40.0
334082,2026-04-30,2026-04-30 12:40:00,2026-04-30 12:43:00,UA,SNA,102.0,2026.0,4.0,3.0,0,...,9.4,83.0,1015.6,3.0,18,2.8,21.363636,36.845295,22.0,3.0
334083,2026-04-30,2026-04-30 12:45:00,2026-04-30 12:45:00,UA,IAD,312.0,2026.0,4.0,3.0,0,...,9.4,83.0,1015.6,12.0,18,2.8,25.714286,39.733038,21.0,0.0
334084,2026-04-30,2026-04-30 12:45:00,2026-04-30 13:09:00,UA,DEN,164.0,2026.0,4.0,3.0,0,...,9.4,83.0,1015.6,2.0,18,2.8,25.714286,39.733038,21.0,24.0


In [6]:
#analyze data

four_df.describe()

,scheduled_elapsed_time_minutes,year,month,day_of_week,is_weekend,sched_dep_hour,outcome,days_until_holiday,days_from_holiday,year_mfr,...,dew_point_temperature,relative_humidity,altimeter,aircraft_age,week_num,temperature_dewpoint_spread,airport_delay_average_1h,airport_delay_stddev_1h,airport_departures_observed_1h,departure_delay_minutes
count,339996.000000,339996.000000,339996.000000,339996.000000,339996.000000,339996.000000,339996.000000,339996.000000,339996.000000,339996.000000,...,339996.000000,339996.000000,339996.000000,339996.000000,339996.000000,339996.000000,339996.000000,339996.000000,339996.000000,339996.000000
mean,217.008135,2023.682176,6.384949,2.973079,0.274812,12.951391,0.511624,45.900308,24.271206,2011.126740,...,9.649758,76.916996,1016.243696,12.555436,25.992944,4.250026,13.573540,22.199175,11.330298,10.284053
std,98.363863,1.189761,3.285837,2.003803,0.446420,5.497408,0.877542,36.679803,19.256173,9.164792,...,3.318651,12.444813,4.690542,9.079588,14.365662,2.853834,37.162049,39.265885,8.420433,40.130938
min,48.000000,2022.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1970.000000,...,-7.800000,10.000000,989.500000,0.000000,1.000000,0.000000,-21.000000,0.000000,0.000000,-92.000000
25%,110.000000,2023.000000,4.000000,1.000000,0.000000,8.000000,0.000000,17.000000,9.000000,2002.000000,...,7.800000,70.000000,1013.200000,5.000000,14.000000,2.200000,-1.400000,2.516611,2.000000,-6.000000
50%,224.000000,2024.000000,6.000000,3.000000,0.000000,12.000000,0.000000,36.000000,19.000000,2014.000000,...,10.000000,79.000000,1015.900000,10.000000,26.000000,3.800000,0.470588,9.671445,12.000000,-2.000000
75%,319.000000,2025.000000,9.000000,5.000000,1.000000,17.000000,1.000000,66.000000,36.000000,2019.000000,...,12.200000,86.000000,1019.000000,21.000000,38.000000,5.500000,14.500000,28.070664,17.000000,9.000000
max,386.000000,2026.000000,12.000000,6.000000,1.000000,23.000000,8.000000,148.000000,74.000000,2026.000000,...,18.900000,100.000000,1034.900000,56.000000,53.000000,35.600000,1158.000000,1120.551606,69.000000,1101.000000


In [7]:
#split train validation and test data

#Do this for 2 hr data
four_train_df = four_df[
    (four_df["date"] >= "2022-01-01") &
    (four_df["date"] <= "2024-12-31")
]

four_validation_df = four_df[
    (four_df["date"] >= "2024-01-01") &
    (four_df["date"] <= "2024-12-31")
]

four_test_df = four_df[
    (four_df["date"] >= "2025-01-01") &
    (four_df["date"] <= "2026-06-09")
]

#Do this for 4 hr data
one_train_df = one_df[
    (one_df["date"] >= "2022-01-01") &
    (one_df["date"] <= "2024-12-31")
]

one_validation_df = one_df[
    (one_df["date"] >= "2024-01-01") &
    (one_df["date"] <= "2024-12-31")
]

one_test_df = one_df[
    (one_df["date"] >= "2025-01-01") &
    (one_df["date"] <= "2026-06-09")
]

#print shape of each
print("2hr data:")
print("Train shape:", four_train_df.shape)
print("Validation shape:", four_validation_df.shape)
print("Test shape:", four_test_df.shape)

print("\n1hr data:")
print("Train shape:", one_train_df.shape)
print("Validation shape:", one_validation_df.shape)
print("Test shape:", one_test_df.shape)

2hr data:
Train shape: (240324, 32)
Validation shape: (86039, 32)
Test shape: (99672, 32)

1hr data:
Train shape: (236106, 32)
Validation shape: (84531, 32)
Test shape: (97980, 32)


In [8]:
four_train_df.columns

Index(['date', 'scheduled_dep_dt', 'actual_dep_dt', 'carrier_code',
       'destination_airport', 'scheduled_elapsed_time_minutes', 'year',
       'month', 'day_of_week', 'is_weekend', 'sched_dep_hour', 'outcome',
       'days_until_holiday', 'days_from_holiday', 'year_mfr', 'model',
       'no_seats', 'visibility', 'ceiling_height', 'wind_speed',
       'wind_direction', 'temperature', 'dew_point_temperature',
       'relative_humidity', 'altimeter', 'aircraft_age', 'week_num',
       'temperature_dewpoint_spread', 'airport_delay_average_1h',
       'airport_delay_stddev_1h', 'airport_departures_observed_1h',
       'departure_delay_minutes'],
      dtype='str')

In [9]:
#feature selection

features = [
    "scheduled_elapsed_time_minutes",
    "year",
    "month",
    "day_of_week",
    "is_weekend",
    "sched_dep_hour",
    "days_until_holiday",
    "days_from_holiday",
    "year_mfr",
    "no_seats",
    "visibility",
    "ceiling_height",
    "wind_speed",
    "wind_direction",
    "temperature",
    "dew_point_temperature",
    "relative_humidity",
    "altimeter",
    "aircraft_age",
    "week_num",
    "temperature_dewpoint_spread",
    "airport_delay_average_1h",
    "airport_delay_stddev_1h",
    "airport_departures_observed_1h"
]

target = "departure_delay_minutes"


**2 hour linear regression baseline**

In [10]:
# create X and y
four_X_train = four_train_df[features]
four_y_train = four_train_df[target]

four_X_val = four_validation_df[features]
four_y_val = four_validation_df[target]

four_X_test = four_test_df[features]
four_y_test = four_test_df[target]

# scale features
scaler = StandardScaler()

four_X_train_scaled = scaler.fit_transform(four_X_train)
four_X_val_scaled = scaler.transform(four_X_val)
four_X_test_scaled = scaler.transform(four_X_test)

In [11]:
#build model
def build_model(num_features, learning_rate):
    tf.keras.backend.clear_session()
    tf.random.set_seed(0)

    model = tf.keras.Sequential([
        tf.keras.Input(shape=(num_features,)),
        tf.keras.layers.Dense(units=1, use_bias=True)
    ])

    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=["mae"]
    )

    return model

In [ ]:
# train model

four_model = build_model(
    num_features=four_X_train_scaled.shape[1],
    learning_rate=0.001
)

four_trained_model = four_model.fit(
    four_X_train_scaled,
    four_y_train,
    validation_data=(four_X_val_scaled, four_y_val),
    epochs=100,
    batch_size=32,
    verbose = 0
)

#plot output
#plot
plt.plot(four_trained_model.history['loss'], label="Training Loss")
plt.plot(four_trained_model.history['val_loss'], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss Values Over Epochs")
plt.legend()
plt.show()


E0000 00:00:1783573944.759347   75331 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1783573944.759700   75389 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1783573944.773590   75331 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [ ]:
#Obtain weight and bias
four_weights, four_bias = four_model.layers[0].get_weights()

#calulate train and test mse

four_train_mse, four_train_mae = four_model.evaluate(
    four_X_train_scaled,
    four_y_train,
    verbose=0
)

# Calculate test MSE and MAE
four_test_mse, four_test_mae = four_model.evaluate(
    four_X_test_scaled,
    four_y_test,
    verbose=0
)

# Learned parameters
print(f"Learned Weights:\n{four_weights}")
print(f"\nLearned Bias:\n{four_bias}")

print(f"Train MSE: {four_train_mse:.3f}")
print(f"Train MAE: {four_train_mae:.3f}")

print(f"Test MSE: {four_test_mse:.3f}")
print(f"Test MAE: {four_test_mae:.3f}")

print(f"Difference between test and train MSE: {abs(four_train_mse - four_test_mse):.3f}")

**1 hour linear regression**

In [ ]:
# create X and y
one_X_train = one_train_df[features]
one_y_train = one_train_df[target]

one_X_val = one_validation_df[features]
one_y_val = one_validation_df[target]

one_X_test = one_test_df[features]
one_y_test = one_test_df[target]

# scale features
scaler = StandardScaler()

one_X_train_scaled = scaler.fit_transform(one_X_train)
one_X_val_scaled = scaler.transform(one_X_val)
one_X_test_scaled = scaler.transform(one_X_test)

In [ ]:
# train model

one_model = build_model(
    num_features=one_X_train_scaled.shape[1],
    learning_rate=0.01
)

one_trained_model = one_model.fit(
    one_X_train_scaled,
    one_y_train,
    validation_data=(one_X_val_scaled, one_y_val),
    epochs=50,
    batch_size=32,
    verbose = 0
)

#plot output
#plot
plt.plot(one_trained_model.history['loss'], label="Training Loss")
plt.plot(one_trained_model.history['val_loss'], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss Values Over Epochs")
plt.legend()
plt.show()

In [ ]:
#Obtain weight and bias
one_weights, one_bias = one_model.layers[0].get_weights()

#calulate train and test mse

one_train_mse, one_train_mae = one_model.evaluate(
    one_X_train_scaled,
    one_y_train,
    verbose=0
)

# Calculate test MSE and MAE
one_test_mse, one_test_mae = one_model.evaluate(
    one_X_test_scaled,
    one_y_test,
    verbose=0
)

# Learned parameters
print(f"Learned Weights:\n{one_weights}")
print(f"\nLearned Bias:\n{one_bias}")

print(f"Train MSE: {one_train_mse:.3f}")
print(f"Train MAE: {one_train_mae:.3f}")

print(f"Test MSE: {one_test_mse:.3f}")
print(f"Test MAE: {one_test_mae:.3f}")

print(f"Difference between test and train MSE: {abs(one_train_mse - one_test_mse):.3f}")

**LOGISTIC REGRESSION**

In [ ]:
#define features
target = "outcome"

numeric_features = [
    "scheduled_elapsed_time_minutes",
    "year",
    "month",
    "day_of_week",
    "is_weekend",
    "sched_dep_hour",
    "days_until_holiday",
    "days_from_holiday",
    "year_mfr",
    "no_seats",
    "visibility",
    "ceiling_height",
    "wind_speed",
    "wind_direction",
    "temperature",
    "dew_point_temperature",
    "relative_humidity",
    "altimeter",
    "aircraft_age",
    "week_num",
    "temperature_dewpoint_spread",
    "airport_delay_average_1h",
    "airport_delay_stddev_1h",
    "airport_departures_observed_1h"
]

categorical_features = [
    "carrier_code",
    "destination_airport",
    "model"
]

**4hr Logistic Regression**

In [ ]:
four_X_train = four_train_df[numeric_features + categorical_features]
four_y_train = four_train_df[target]

four_X_val = four_validation_df[numeric_features + categorical_features]
four_y_val = four_validation_df[target]

four_X_test = four_test_df[numeric_features + categorical_features]
four_y_test = four_test_df[target]

In [ ]:
#build preprocessing pipeline
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

four_X_train_processed = preprocessor.fit_transform(four_X_train)

four_X_val_processed = preprocessor.transform(four_X_val)

four_X_test_processed = preprocessor.transform(four_X_test)

#check sahpe

print("Train shape:", four_X_train_processed.shape)
print("Validation shape:", four_X_val_processed.shape)
print("Test shape:", four_X_test_processed.shape)

In [ ]:
#build model

def build_model(num_features, learning_rate):
    """Return a simple multiclass logistic regression model using Keras."""

    tf.keras.backend.clear_session()
    tf.random.set_seed(0)

    model = tf.keras.Sequential()

    model.add(tf.keras.Input(shape=(num_features,), name="Input"))

    model.add(tf.keras.layers.Dense(
        units=9,
        use_bias=True,
        activation="softmax",
        kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01),
        bias_initializer=tf.keras.initializers.RandomNormal(stddev=0.01),
        name="Output"
    ))

    model.compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
        metrics=["accuracy"]
    )

    return model

In [ ]:
#train

four_model = build_model(
    num_features=four_X_train_processed.shape[1],
    learning_rate=0.01
)

four_trained_model = four_model.fit(
    four_X_train_processed,
    four_y_train,
    validation_data=(four_X_val_processed, four_y_val),
    epochs=50,
    batch_size=32,
    verbose=0
)

In [ ]:
#plot

# Get number of epochs actually trained
num_epochs = len(four_trained_model.history["loss"])
epochs = range(1, num_epochs + 1)

plt.figure(figsize=(12, 5))

# Subplot 1: Loss
plt.subplot(1, 2, 1)
plt.plot(epochs, four_trained_model.history["loss"], label="Training Loss")
plt.plot(epochs, four_trained_model.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid()

# Subplot 2: Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, four_trained_model.history["accuracy"], label="Training Accuracy")
plt.plot(epochs, four_trained_model.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid()

plt.tight_layout()
plt.show()

In [ ]:
four_train_loss, four_train_acc = four_model.evaluate(
    four_X_train_processed,
    four_y_train,
    verbose=0
)

four_val_loss, four_val_acc = four_model.evaluate(
    four_X_val_processed,
    four_y_val,
    verbose=0
)

four_test_loss, four_test_acc = four_model.evaluate(
    four_X_test_processed,
    four_y_test,
    verbose=0
)

print(f"Train Loss: {four_train_loss:.3f}")
print(f"Train Accuracy: {four_train_acc:.3f}")

print(f"\nValidation Loss: {four_val_loss:.3f}")
print(f"Validation Accuracy: {four_val_acc:.3f}")

print(f"\nTest Loss: {four_test_loss:.3f}")
print(f"Test Accuracy: {four_test_acc:.3f}")